In [3]:
# ==============================================================
# Sales Forecast v6 — OMS Edition (Ridge-NNLS + Smooth Adaptive + OMS Scenarios) + V6.1 Bootstrap Patched
# - Exog Methods: Prophet, SARIMA, ETS, ML-RF, ML-XGB
# - Ensembles:
#    * Inverse-MAE: All-5-INV, Top2-INV, Top3-INV
#    * NNLS/Stacking: All-5-NNLS, Top2-NNLS, Top3-NNLS  (+ Ridge versiyonları)
#    * Adaptive NNLS (rolling): Adaptive5-NNLS-w{3,4,6}  (+ Ridge + |Δw| ceza)
# - Y Variants: RF, XGB, Y-ENS (RF+XGB; VAL tabanlı inverse-MAE w)
# - Bootstrap PI (%80/%95), stok-tükeniş riski, OMS sipariş önerisi
# - Tüm özetler hem dosyaya yazılır hem konsola basılır
# ==============================================================

import warnings, os, sys, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product
from collections import OrderedDict, defaultdict

from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Optional SHAP
try:
    import shap
    HAVE_SHAP = True
except Exception:
    HAVE_SHAP = False

from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# ------------------ Config ------------------
CSV_PATH   = "veri_matrisi_final_sales_orders_stock_calendar_lags_fx.csv"

VAL_START  = pd.Timestamp("2024-07-01")
VAL_END    = pd.Timestamp("2024-12-01")
TEST_START = pd.Timestamp("2025-01-01")
TEST_END   = pd.Timestamp("2025-07-01")
TEST_END_SHORT = pd.Timestamp("2025-03-01")

RANDOM_STATE = 42
EXOG_VAL_H   = 6
EPS_PROPHET  = 0.05
B_BOOT       = 800
SEED         = 1337

FEATURES_Y = [
    "orders","stock",
    "orders_lag1","orders_lag3",
    "stock_lag1","stock_lag3",
    "y_lag1",
    "orders_ratio",
    "month","year",
]

STARTING_STOCK_OVERRIDE = None

# ---- OMS policy (parametrik) ----
T_CHECK_LIST  = [3]
H_COVER_LIST  = [6]
COVER_Q_LIST  = [0.50, 0.80, 0.90]
MOQ          = 0.0
LOT_SIZE     = 1.0

# ---- Adaptive NNLS window ve cezalar ----
ADAPT_WINS   = [3, 4, 6]
RIDGE_ALPHA  = 1e-3
SMOOTH_BETA  = 0.15

# ------------------ Utils ------------------
rng_boot = np.random.default_rng(SEED)

def mae_rmse_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(np.mean((y_pred - y_true)**2))
    denom = np.where(y_true == 0, 1, y_true)
    mape = np.mean(np.abs((y_true - y_pred) / denom)) * 100
    return mae, rmse, mape

def ensure_ms_freq(df):
    d = df.copy().sort_values("ds")
    d["ds"] = d["ds"].dt.to_period("M").dt.to_timestamp(how="start")
    d = d.set_index("ds").sort_index()
    d.index = pd.DatetimeIndex(d.index, freq="MS")
    return d.reset_index()

def add_calendar(df):
    d = df.copy()
    d["year"]  = d["ds"].dt.year
    d["month"] = d["ds"].dt.month
    return d

def rolling_impute(s, causal=False):
    x = pd.to_numeric(s, errors="coerce")
    if causal:
        x = x.ffill()
        x = x.rolling(window=3, min_periods=1).mean()
        x = x.bfill()
    else:
        roll = x.rolling(window=3, center=True, min_periods=1).mean()
        x = x.where(~x.isna(), roll).ffill().bfill()
    return x

def smooth_causal_ma(s, window=3):
    x = pd.to_numeric(s, errors="coerce").ffill()
    return x.rolling(window=window, min_periods=1).mean().bfill()

def winsorize_series(s, lower_q=0.05, upper_q=0.95):
    x = pd.to_numeric(s, errors="coerce")
    lo = np.nanpercentile(x, lower_q*100)
    hi = np.nanpercentile(x, upper_q*100)
    return x.clip(lo, hi)

def nonneg(s):
    return pd.to_numeric(s, errors="coerce").clip(lower=0.0)

def build_lags_y(df):
    d = df.copy()
    if "orders" in d.columns and "stock" in d.columns:
        d["orders_ratio"] = d["orders"] / d["stock"].replace(0, np.nan)
    if "y" in d.columns:
        d["y_lag1"] = d["y"].shift(1)
    if "orders" in d.columns:
        d["orders_lag1"] = d["orders"].shift(1)
        d["orders_lag3"] = d["orders"].shift(3)
    if "stock" in d.columns:
        d["stock_lag1"] = d["stock"].shift(1)
        d["stock_lag3"] = d["stock"].shift(3)
    return d

def prep_features_y(df_in, causal=False):
    d = add_calendar(df_in)
    d = build_lags_y(d)
    for col in ["orders","stock"]:
        if col in d.columns:
            d[col] = rolling_impute(d[col], causal=causal)
    for col in ["orders_lag1","orders_lag3","stock_lag1","stock_lag3","y_lag1","orders_ratio"]:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce").ffill().bfill().fillna(0.0)
    for c in FEATURES_Y:
        if c not in d.columns:
            d[c] = 0.0
    return d.replace([np.inf, -np.inf], np.nan).fillna(0)

# ------------------ Univariate exog models ------------------
def fit_prophet(train_df, value_col):
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False)
    m.fit(train_df.rename(columns={value_col:"y"}))
    return m

def forecast_prophet(model, steps):
    fut = model.make_future_dataframe(periods=steps, freq="MS")
    fc  = model.predict(fut)[["ds","yhat"]].tail(steps)
    return fc.rename(columns={"yhat":"yhat"})

def sarima_fit_best(y, p_range=(0,3), q_range=(0,3), P_range=(0,1), Q_range=(0,1)):
    best, best_aic = None, np.inf
    for p in range(p_range[0], p_range[1]+1):
        for q in range(q_range[0], q_range[1]+1):
            for P in range(P_range[0], P_range[1]+1):
                for Q in range(Q_range[0], Q_range[1]+1):
                    try:
                        r = SARIMAX(y, order=(p,1,q), seasonal_order=(P,1,Q,12),
                                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
                        if r.aic < best_aic:
                            best_aic = r.aic
                            best = ((p,1,q),(P,1,Q,12))
                    except Exception:
                        pass
    return best

def fit_sarima(train_df, value_col):
    y = train_df.set_index("ds")[value_col]
    y.index.freq = "MS"
    best = sarima_fit_best(y)
    if best is None:
        return None
    return SARIMAX(y, order=best[0], seasonal_order=best[1],
                   enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

def forecast_sarima(model, steps, future_idx):
    pred = model.get_forecast(steps=steps).predicted_mean
    return pd.DataFrame({"ds": pd.DatetimeIndex(future_idx), "yhat": pred.values})

def fit_ets(train_df, value_col):
    y = train_df.set_index("ds")[value_col]
    y.index.freq = "MS"
    best_model, best_aic = None, np.inf
    for trend in ["add", "mul", None]:
        for seasonal in ["add", "mul", None]:
            try:
                for damped in [True, False]:
                    if seasonal is None:
                        model = ExponentialSmoothing(y, trend=trend, seasonal=None, damped_trend=damped).fit(optimized=True)
                    else:
                        model = ExponentialSmoothing(y, trend=trend, seasonal=seasonal, seasonal_periods=12,
                                                     damped_trend=damped).fit(optimized=True)
                    aic = getattr(model, "aic", np.inf)
                    if aic < best_aic:
                        best_aic, best_model = aic, model
            except Exception:
                continue
    if best_model is None:
        best_model = ExponentialSmoothing(y, trend="add", seasonal="add", seasonal_periods=12).fit(optimized=True)
    return best_model

def forecast_ets(model, steps, future_idx):
    pred = model.forecast(steps)
    return pd.DataFrame({"ds": pd.DatetimeIndex(future_idx), "yhat": pred.values})

def backtest_mae(series_df, value_col, method, cutoff, val_h=EXOG_VAL_H):
    s_full = series_df[["ds", value_col]].dropna().sort_values("ds")
    s = s_full[s_full["ds"] < cutoff]
    if len(s) < val_h + 6:
        return np.inf
    cut = s["ds"].max() - pd.DateOffset(months=val_h-1)
    train = s[s["ds"] < cut]
    val   = s[s["ds"] >= cut]
    steps = len(val)
    try:
        if method == "prophet":
            m = fit_prophet(train, value_col)
            yhat = forecast_prophet(m, steps)["yhat"].values
        elif method == "sarima":
            m = fit_sarima(train, value_col)
            if m is None: return np.inf
            yhat = forecast_sarima(m, steps, val["ds"].values)["yhat"].values
        else:
            m = fit_ets(train, value_col)
            yhat = forecast_ets(m, steps, val["ds"].values)["yhat"].values
    except Exception:
        return np.inf
    return mae_rmse_mape(val[value_col].values, yhat)[0]

def _postprocess_exog(df):
    d = df.copy()
    for c in ["orders","stock"]:
        if c in d.columns:
            d[c] = smooth_causal_ma(d[c], window=3)
            d[c] = winsorize_series(d[c], 0.05, 0.95)
            d[c] = nonneg(d[c])
    return d

# ------------------ ML-Exog (orders/stock) ------------------
EXOG_FEATURES = ["lag1","lag3","month","year"]

def make_exog_feature_frame(df_var, var_col):
    d = df_var[["ds", var_col]].sort_values("ds").copy()
    d["lag1"] = d[var_col].shift(1)
    d["lag3"] = d[var_col].shift(3)
    d["month"] = d["ds"].dt.month
    d["year"]  = d["ds"].dt.year
    return d

def train_exog_rf(df_var, var_col, cutoff):
    d = make_exog_feature_frame(df_var[df_var["ds"] < cutoff], var_col).dropna()
    if d.empty: return None
    X = d[EXOG_FEATURES]; y = d[var_col]
    rf = RandomForestRegressor(n_estimators=400, max_depth=8, min_samples_split=2, min_samples_leaf=1,
                               random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X, y); return rf

def train_exog_xgb(df_var, var_col, cutoff):
    d = make_exog_feature_frame(df_var[df_var["ds"] < cutoff], var_col).dropna()
    if d.empty: return None
    X = d[EXOG_FEATURES].to_numpy(); y = d[var_col].to_numpy()
    xgb = XGBRegressor(n_estimators=500, learning_rate=0.08, max_depth=3,
                       subsample=0.9, colsample_bytree=0.9, reg_lambda=1.2,
                       random_state=RANDOM_STATE)
    xgb.fit(X, y, verbose=False); return xgb

def recursive_forecast_exog(model, hist_df, var_col, start_ds, end_ds):
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    full = hist_df[["ds", var_col]].sort_values("ds").copy()
    out = []
    for ds in future_idx:
        tmp = make_exog_feature_frame(full, var_col)
        row = tmp[tmp["ds"] == ds][EXOG_FEATURES]
        if row.empty:
            row = pd.DataFrame({
                "lag1": [full[var_col].iloc[-1] if len(full) else 0.0],
                "lag3": [full[var_col].iloc[-3] if len(full) >= 3 else (full[var_col].iloc[-1] if len(full) else 0.0)],
                "month":[ds.month], "year":[ds.year]
            })
        x = row.to_numpy()
        yhat = float(model.predict(x)[0])
        out.append({"ds": ds, var_col: max(0.0, yhat)})
        full = pd.concat([full, pd.DataFrame({"ds":[ds], var_col:[yhat]})], ignore_index=True)
    return pd.DataFrame(out)

# ------------------ Exog builders ------------------
def build_exog_univariate_for_period(df_all, start_ds, end_ds, cutoff, uni_method):
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    out = pd.DataFrame({"ds": future_idx})
    for col in ["orders","stock"]:
        s = df_all[["ds", col]].dropna().sort_values("ds")
        s = s[s["ds"] < cutoff]
        if s.empty:
            out[col] = 0.0; continue
        steps = len(future_idx)
        try:
            if uni_method == "prophet":
                m = fit_prophet(s, col);  fc = forecast_prophet(m, steps)
            elif uni_method == "sarima":
                m = fit_sarima(s, col);   fc = forecast_sarima(m, steps, future_idx)
            else:
                m = fit_ets(s, col);      fc = forecast_ets(m, steps, future_idx)
            tmp = fc.rename(columns={"yhat":col})
        except Exception:
            tmp = pd.DataFrame({"ds": future_idx, col: np.nan})
        out = out.merge(tmp[["ds",col]], on="ds", how="left")
    return _postprocess_exog(out)

def build_exog_inverse_mae_for_period(df_all, start_ds, end_ds, cutoff, eps_prophet=EPS_PROPHET):
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    out = pd.DataFrame({"ds": future_idx})
    for col in ["orders","stock"]:
        s = df_all[["ds", col]].dropna().sort_values("ds")
        s = s[s["ds"] < cutoff]
        if s.empty:
            out[col] = 0.0; continue
        steps = len(future_idx)
        mae_p = backtest_mae(s, col, "prophet", cutoff)
        mae_s = backtest_mae(s, col, "sarima",  cutoff)
        mae_e = backtest_mae(s, col, "ets",     cutoff)
        try: mp  = fit_prophet(s, col); fcp = forecast_prophet(mp, steps).rename(columns={"yhat":"p"})
        except Exception: fcp = pd.DataFrame({"ds": future_idx, "p": np.nan})
        try: ms  = fit_sarima(s, col);  fcs = forecast_sarima(ms, steps, future_idx).rename(columns={"yhat":"s"})
        except Exception: fcs = pd.DataFrame({"ds": future_idx, "s": np.nan})
        try: me  = fit_ets(s, col);     fce = forecast_ets(me, steps, future_idx).rename(columns={"yhat":"e"})
        except Exception: fce = pd.DataFrame({"ds": future_idx, "e": np.nan})

        maes = np.array([mae_p, mae_s, mae_e], dtype=float)
        maes = np.where(~np.isfinite(maes) | (maes <= 0), np.nan, maes)
        inv  = 1.0 / np.where(np.isnan(maes), np.inf, maes)
        if np.isfinite(inv).sum() == 0:
            wp, ws, we = 0.60, 0.25, 0.15
        else:
            w = inv / inv.sum()
            w[0] += eps_prophet
            w = w / w.sum()
            wp, ws, we = w[0], w[1], w[2]

        tmp = (pd.DataFrame({"ds": future_idx})
                 .merge(fcp, on="ds", how="left")
                 .merge(fcs, on="ds", how="left")
                 .merge(fce, on="ds", how="left"))
        tmp[col] = (wp*tmp["p"] + ws*tmp["s"] + we*tmp["e"])
        out = out.merge(tmp[["ds",col]], on="ds", how="left")
    return _postprocess_exog(out)

def build_exog_ml_for_period(df_all, start_ds, end_ds, cutoff, learner="rf"):
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    out = pd.DataFrame({"ds": future_idx})
    for col in ["orders","stock"]:
        s_hist = df_all[df_all["ds"] < cutoff][["ds", col]].copy()
        if s_hist.empty:
            out[col] = 0.0; continue
        mdl = train_exog_rf(df_all[["ds", col]].copy(), col, cutoff) if learner=="rf" \
              else train_exog_xgb(df_all[["ds", col]].copy(), col, cutoff)
        if mdl is None:
            out[col] = 0.0; continue
        fc = recursive_forecast_exog(mdl, s_hist, col, start_ds, end_ds)
        out = out.merge(fc, on="ds", how="left")
    return _postprocess_exog(out)

# ------------------ NNLS solvers ------------------
def project_simplex(w, eps=1e-12):
    u = np.sort(np.maximum(w,0))[::-1]
    cssv = np.cumsum(u)
    rho = np.where(u > (cssv - 1) / (np.arange(len(u)) + 1))[0]
    if len(rho)==0: return np.ones_like(w)/len(w)
    rho = rho[-1]
    theta = (cssv[rho] - 1) / (rho + 1.0)
    w = np.maximum(w - theta, 0)
    s = w.sum()
    if s <= eps: w[:] = 1.0/len(w)
    else: w /= s
    return w

def nnls_ridge(A, y, alpha=0.0, iters=800, eps=1e-8):
    A = np.asarray(A, float); y = np.asarray(y, float)
    T, K = A.shape
    if K==1: return np.array([1.0])
    L = np.linalg.norm(A, ord=2)**2 + alpha + 1e-6
    step = 1.0 / L
    w = np.ones(K)/K
    AT = A.T
    for _ in range(iters):
        grad = 2*(AT@(A@w - y)) + 2*alpha*w
        w = w - step*grad
        w = project_simplex(w)
    return w

def nnls_adaptive_update(A, y, w_prev, alpha=0.0, beta=0.0, iters=400):
    A = np.asarray(A, float); y = np.asarray(y, float)
    T, K = A.shape
    if K==1: return np.array([1.0])
    L = np.linalg.norm(A, ord=2)**2 + alpha + beta + 1e-6
    step = 1.0 / L
    w = w_prev.copy() if w_prev is not None else np.ones(K)/K
    AT = A.T
    for _ in range(iters):
        grad = 2*(AT@(A@w - y)) + 2*alpha*w + 2*beta*(w - (w_prev if w_prev is not None else 0))
        w = project_simplex(w - step*grad)
    return w

def combine_exogs_weighted(exog_dict, weights_by_var):
    names = list(next(iter(weights_by_var.values())).keys())
    base = exog_dict[names[0]][["ds"]].copy()
    out = base.copy()
    for var in ["orders","stock"]:
        s = np.zeros(len(base), dtype=float)
        for nm, w in weights_by_var[var].items():
            s += float(w) * pd.to_numeric(exog_dict[nm][var].values, errors="coerce")
        out[var] = s
    return _postprocess_exog(out)

# ------------------ ROCV on Y ------------------
def rolling_origin_splits(df, n_splits=3, min_train_months=24):
    d = df.sort_values("ds").copy()
    if d["ds"].nunique() < (min_train_months + n_splits):
        yield (d[d["ds"] < d["ds"].max() - pd.DateOffset(months=3)],
               d[d["ds"] >= d["ds"].max() - pd.DateOffset(months=3)])
        return
    for k in range(n_splits, 0, -1):
        val_end = d["ds"].max() - pd.DateOffset(months=k-1)
        val_start = val_end - pd.DateOffset(months=2)
        train_end = val_start - pd.DateOffset(days=1)
        train = d[(d["ds"] <= train_end)]
        val   = d[(d["ds"] >= val_start) & (d["ds"] <= val_end)]
        if len(train) >= min_train_months and len(val) >= 2:
            yield train, val

def optimize_rf_rocv(df_trainval):
    base_grid = {
        "n_estimators": [400, 700],
        "max_depth": [None, 8, 12],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2],
    }
    best, best_score = None, np.inf
    for params in (dict(zip(base_grid.keys(), v)) for v in product(*base_grid.values())):
        maes = []
        for tr, va in rolling_origin_splits(df_trainval, n_splits=3, min_train_months=24):
            mdl = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
            mdl.fit(tr[FEATURES_Y], tr["y"])
            pred = mdl.predict(va[FEATURES_Y])
            maes.append(mean_absolute_error(va["y"], pred))
        score = np.mean(maes) if maes else np.inf
        if score < best_score:
            best_score, best = score, params
    final = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best).fit(df_trainval[FEATURES_Y], df_trainval["y"])
    return final, best, best_score

def optimize_xgb_rocv(df_trainval):
    base_grid = {
        "n_estimators": [500, 800],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 4],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "reg_lambda": [1.0, 2.0],
    }
    best, best_score = None, np.inf
    for params in (dict(zip(base_grid.keys(), v)) for v in product(*base_grid.values())):
        maes = []
        for tr, va in rolling_origin_splits(df_trainval, n_splits=3, min_train_months=24):
            mdl = XGBRegressor(random_state=RANDOM_STATE, **params)
            mdl.fit(tr[FEATURES_Y].to_numpy(), tr["y"].to_numpy(), verbose=False)
            pred = mdl.predict(va[FEATURES_Y].to_numpy())
            maes.append(mean_absolute_error(va["y"], pred))
        score = np.mean(maes) if maes else np.inf
        if score < best_score:
            best_score, best = score, params
    final = XGBRegressor(random_state=RANDOM_STATE, **best)
    final.fit(df_trainval[FEATURES_Y].to_numpy(), df_trainval["y"].to_numpy(), verbose=False)
    return final, best, best_score

# ------------------ Y recursive forward ------------------
def recursive_forward_predict_y(model, x_cols, hist_df, future_exog, start_ds, end_ds):
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    future_part = pd.DataFrame({"ds": future_idx}).merge(future_exog, on="ds", how="left")
    full = pd.concat([hist_df, future_part], ignore_index=True).sort_values("ds")

    preds = []
    for ds in future_idx:
        tmp = prep_features_y(full.copy(), causal=True)
        row = tmp.loc[tmp["ds"] == ds].copy()
        X = row[x_cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()
        y_hat = float(model.predict(X)[0])
        preds.append({"ds": ds, "yhat": y_hat})
        full.loc[full["ds"] == ds, "y"] = y_hat
        for c in ["orders","stock"]:
            if c in full.columns:
                full[c] = rolling_impute(full[c], causal=True)
    preds_df = pd.DataFrame(preds)
    used_future = full.loc[full["ds"].isin(future_idx)].copy()
    return preds_df, used_future

# ------------------ Load & prepare ------------------
os.makedirs("outputs", exist_ok=True)
df_raw = pd.read_csv(CSV_PATH, parse_dates=["ds"]).sort_values("ds").reset_index(drop=True)
for c in ["y","orders","stock"]:
    if c in df_raw.columns: df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce")
df_raw = ensure_ms_freq(df_raw)

mask_train = (df_raw["ds"] < VAL_START)
mask_val   = (df_raw["ds"] >= VAL_START) & (df_raw["ds"] <= VAL_END)

train_df = prep_features_y(df_raw.loc[mask_train].copy(), causal=False)
val_df   = prep_features_y(df_raw.loc[mask_val].copy(),   causal=True)
trainval_df = pd.concat([train_df, val_df], ignore_index=True)

# Y models
rf_model,  rf_params,  rf_rocv_mae  = optimize_rf_rocv(trainval_df)
xgb_model, xgb_params, xgb_rocv_mae = optimize_xgb_rocv(trainval_df)

print("\n=== ROCV Best Params ===")
print("RF :", rf_params,  f"| ROCV_MAE={rf_rocv_mae:.2f}")
print("XGB:", xgb_params, f"| ROCV_MAE={xgb_rocv_mae:.2f}")

# ------------------ VAL exogs (basic) ------------------
def build_exog(method, df_all, start_ds, end_ds, cutoff):
    if method == "Prophet":
        return build_exog_univariate_for_period(df_all, start_ds, end_ds, cutoff, "prophet")
    if method == "SARIMA":
        return build_exog_univariate_for_period(df_all, start_ds, end_ds, cutoff, "sarima")
    if method == "ETS":
        return build_exog_univariate_for_period(df_all, start_ds, end_ds, cutoff, "ets")
    if method == "Ensemble":
        return build_exog_inverse_mae_for_period(df_all, start_ds, end_ds, cutoff)
    if method == "ML-Exog RF":
        return build_exog_ml_for_period(df_all, start_ds, end_ds, cutoff, "rf")
    if method == "ML-Exog XGB":
        return build_exog_ml_for_period(df_all, start_ds, end_ds, cutoff, "xgb")
    raise ValueError("Unknown basic exog method")

BASIC = ["Prophet","SARIMA","ETS","Ensemble","ML-Exog RF","ML-Exog XGB"]
exog_val = {m: build_exog(m, df_raw[["ds","orders","stock"]], VAL_START, VAL_END, cutoff=VAL_START) for m in BASIC}

# ------------------ VAL scoring helper ------------------
def y_ensemble_weights(y_true, yhat_rf, yhat_xgb, eps=1e-6):
    mae_rf  = mean_absolute_error(y_true, yhat_rf)
    mae_xgb = mean_absolute_error(y_true, yhat_xgb)
    w_rf, w_xgb = 1.0/(mae_rf+eps), 1.0/(mae_xgb+eps)
    s = w_rf + w_xgb
    return w_rf/s, w_xgb/s, mae_rf, mae_xgb

def eval_exog_on_val(exog_tbl):
    hist_for_val = train_df[["ds","y","orders","stock","month","year"]].copy()
    prf,_  = recursive_forward_predict_y(rf_model,  FEATURES_Y, hist_for_val.copy(), exog_tbl, VAL_START, VAL_END)
    pxgb,_ = recursive_forward_predict_y(xgb_model, FEATURES_Y, hist_for_val.copy(), exog_tbl, VAL_START, VAL_END)
    join = val_df[["ds","y"]].merge(prf, on="ds", how="left").rename(columns={"yhat":"yhat_rf"}) \
                             .merge(pxgb, on="ds", how="left").rename(columns={"yhat":"yhat_xgb"})
    w_rf, w_xgb, mae_rf, mae_xgb = y_ensemble_weights(join["y"].values, join["yhat_rf"].values, join["yhat_xgb"].values)
    yhat_ens = w_rf*join["yhat_rf"].values + w_xgb*join["yhat_xgb"].values
    mae_ens, rmse_ens, mape_ens = mae_rmse_mape(join["y"].values, yhat_ens)
    return {
        "weights": (w_rf, w_xgb),
        "mae_rf": mae_rf, "mae_xgb": mae_xgb,
        "mae_ens": mae_ens, "rmse_ens": rmse_ens, "mape_ens": mape_ens,
        "residuals": {"RF": (join["y"].values - join["yhat_rf"].values),
                      "XGB": (join["y"].values - join["yhat_xgb"].values),
                      "ENS": (join["y"].values - yhat_ens)}
    }, join

val_rep = {}
for m in BASIC:
    rep,_ = eval_exog_on_val(exog_val[m]); val_rep[m] = rep

def val_summary_table(rep_dict):
    rows = []
    for m,rep in rep_dict.items():
        w_rf, w_xgb = rep["weights"]
        rows.append([m, rep["mae_rf"], rep["mae_xgb"], rep["mae_ens"], rep["rmse_ens"], rep["mape_ens"], w_rf, w_xgb])
    d = pd.DataFrame(rows, columns=["Exog","VAL_MAE_RF","VAL_MAE_XGB","VAL_MAE_YENS","VAL_RMSE_YENS","VAL_MAPE_YENS","w_RF","w_XGB"]) \
          .sort_values("VAL_MAE_YENS")
    return d

val_table_basic = val_summary_table(val_rep)
val_table_basic.to_csv("outputs/val_exog_selection_basic.csv", index=False)
print("\n=== VAL Exog Selection (BASIC methods) ===")
print(val_table_basic.to_string(index=False))

# ------------------ Inverse & NNLS (static) on VAL ------------------
ALL5 = ["Prophet","SARIMA","ETS","ML-Exog RF","ML-Exog XGB"]

mae_all5 = {m: val_rep[m]["mae_ens"] for m in ALL5}
ranked = sorted(ALL5, key=lambda m: (mae_all5[m], m))
Top2 = ranked[:2]; Top3 = ranked[:3]

def inv_weights(methods):
    inv = []
    for m in methods:
        v = mae_all5[m]
        inv.append(0.0 if (not np.isfinite(v) or v<=0) else 1.0/v)
    s = sum(inv) if sum(inv)>0 else 1.0
    return {m: inv[i]/s for i,m in enumerate(methods)}

inv_all5 = inv_weights(ALL5)
inv_top2 = inv_weights(Top2)
inv_top3 = inv_weights(Top3)

# >>>>> FIXED FUNCTION (no overlapping merge; aligned matrices) <<<<<
def fit_nnls_weights_on_val(exog_dict, df_true, alpha=0.0):
    """
    exog_dict: {name: DataFrame[ds, orders, stock]}  (VAL dönemine ait)
    df_true:   DataFrame[ds, orders, stock]          (VAL gerçekleri)
    Döndürür:  {"orders": {name: w_i}, "stock": {name: w_i}}
    """
    names = list(exog_dict.keys())
    base_ds = df_true[["ds"]].copy()

    # Her yöntem için VAL ds'e hizala
    A_o_cols, A_s_cols = [], []
    for nm in names:
        tmp = base_ds.merge(exog_dict[nm][["ds","orders","stock"]], on="ds", how="left")
        o = pd.to_numeric(tmp["orders"], errors="coerce")
        s = pd.to_numeric(tmp["stock"],  errors="coerce")
        # VAL periyodunda eksik varsa güvenli doldurma
        o = o.ffill().bfill().fillna(0.0).to_numpy()
        s = s.ffill().bfill().fillna(0.0).to_numpy()
        A_o_cols.append(o)
        A_s_cols.append(s)

    # Tasarım matrisi: T x K
    A_o = np.column_stack(A_o_cols)
    A_s = np.column_stack(A_s_cols)

    y_o = pd.to_numeric(df_true["orders"], errors="coerce").ffill().bfill().fillna(0.0).to_numpy()
    y_s = pd.to_numeric(df_true["stock"],  errors="coerce").ffill().bfill().fillna(0.0).to_numpy()

    w_o = nnls_ridge(A_o, y_o, alpha=alpha)
    w_s = nnls_ridge(A_s, y_s, alpha=alpha)

    return {
        "orders": {names[i]: float(w_o[i]) for i in range(len(names))},
        "stock":  {names[i]: float(w_s[i]) for i in range(len(names))}
    }
# <<<<< END FIX >>>>>

def combine_by_inv(methods, inv_w):
    return {"orders":{m:inv_w[m] for m in methods}, "stock":{m:inv_w[m] for m in methods}}

def build_and_score_composite_on_val(tag, weights_by_var, parts):
    ex = combine_exogs_weighted(parts, weights_by_var)
    rep,_ = eval_exog_on_val(ex)
    return tag, ex, rep

val_exog_truth = df_raw[(df_raw["ds"]>=VAL_START)&(df_raw["ds"]<=VAL_END)][["ds","orders","stock"]].reset_index(drop=True)

# NNLS statik (cezasız) ve Ridge
nnls_all5       = fit_nnls_weights_on_val({m:exog_val[m] for m in ALL5}, val_exog_truth, alpha=0.0)
nnls_top2       = fit_nnls_weights_on_val({m:exog_val[m] for m in Top2}, val_exog_truth, alpha=0.0)
nnls_top3       = fit_nnls_weights_on_val({m:exog_val[m] for m in Top3}, val_exog_truth, alpha=0.0)
nnls_all5_ridge = fit_nnls_weights_on_val({m:exog_val[m] for m in ALL5}, val_exog_truth, alpha=RIDGE_ALPHA)
nnls_top2_ridge = fit_nnls_weights_on_val({m:exog_val[m] for m in Top2}, val_exog_truth, alpha=RIDGE_ALPHA)
nnls_top3_ridge = fit_nnls_weights_on_val({m:exog_val[m] for m in Top3}, val_exog_truth, alpha=RIDGE_ALPHA)

def build_and_collect(tag, wv, parts):
    t, ex, rep = build_and_score_composite_on_val(tag, wv, parts)
    return t, ex, rep

composite_parts_val = {m: exog_val[m] for m in ALL5}

# inverse tabanlı VAL skorları
rep_add = {}
tag_ex_val = {}
for tag, wv in [("All-5-INV", combine_by_inv(ALL5, inv_all5)),
                ("Top2-INV",  combine_by_inv(Top2, inv_top2)),
                ("Top3-INV",  combine_by_inv(Top3, inv_top3))]:
    t, ex, rep = build_and_collect(tag, wv, composite_parts_val)
    tag_ex_val[tag] = ex; rep_add[tag] = rep

# NNLS tabanlı (cezasız + ridge)
tag_nnls = {}
for tag, wv in [("All-5-NNLS", nnls_all5), ("Top2-NNLS", nnls_top2), ("Top3-NNLS", nnls_top3),
                ("All-5-NNLS-Ridge", nnls_all5_ridge), ("Top2-NNLS-Ridge", nnls_top2_ridge), ("Top3-NNLS-Ridge", nnls_top3_ridge)]:
    t, ex, rep = build_and_collect(tag, wv, composite_parts_val)
    tag_nnls[tag] = ex; rep_add[tag] = rep

# VAL raporu (tümü)
val_all_reports = {**val_rep, **rep_add}
val_table_all = val_summary_table(val_all_reports)
val_table_all.to_csv("outputs/val_exog_selection_ALL_with_NNLS_Ridge.csv", index=False)

print("\n=== VAL Exog Selection — Inverse + NNLS (Ridge dahil) ===")
print(val_table_all.to_string(index=False))

# ------------------ Build TEST exogs for ALL variants (incl. adaptive) ------------------
def build_test_basic(method):
    df_all = df_raw[["ds","orders","stock"]]
    full  = build_exog(method, df_all, TEST_START, TEST_END,        cutoff=TEST_START)
    short = build_exog(method, df_all, TEST_START, TEST_END_SHORT,  cutoff=TEST_START)
    return full, short

exog_test_full = {}
exog_test_short= {}

for m in BASIC:
    f,s = build_test_basic(m); exog_test_full[m]=f; exog_test_short[m]=s

def build_test_composite_from_weights(weights_by_var):
    methods = list(weights_by_var["orders"].keys())
    parts_full  = {m: exog_test_full[m]  for m in methods}
    parts_short = {m: exog_test_short[m] for m in methods}
    full  = combine_exogs_weighted(parts_full,  weights_by_var)
    short = combine_exogs_weighted(parts_short, weights_by_var)
    return full, short

# inverse composites
exog_test_full["All-5-INV"],  exog_test_short["All-5-INV"]  = build_test_composite_from_weights(combine_by_inv(ALL5, inv_all5))
exog_test_full["Top2-INV"],   exog_test_short["Top2-INV"]   = build_test_composite_from_weights(combine_by_inv(Top2, inv_top2))
exog_test_full["Top3-INV"],   exog_test_short["Top3-INV"]   = build_test_composite_from_weights(combine_by_inv(Top3, inv_top3))

# NNLS composites (statik)
exog_test_full["All-5-NNLS"],       exog_test_short["All-5-NNLS"]       = build_test_composite_from_weights(nnls_all5)
exog_test_full["Top2-NNLS"],        exog_test_short["Top2-NNLS"]        = build_test_composite_from_weights(nnls_top2)
exog_test_full["Top3-NNLS"],        exog_test_short["Top3-NNLS"]        = build_test_composite_from_weights(nnls_top3)
exog_test_full["All-5-NNLS-Ridge"], exog_test_short["All-5-NNLS-Ridge"] = build_test_composite_from_weights(nnls_all5_ridge)
exog_test_full["Top2-NNLS-Ridge"],  exog_test_short["Top2-NNLS-Ridge"]  = build_test_composite_from_weights(nnls_top2_ridge)
exog_test_full["Top3-NNLS-Ridge"],  exog_test_short["Top3-NNLS-Ridge"]  = build_test_composite_from_weights(nnls_top3_ridge)

# ------------------ Adaptive NNLS (rolling monthly, ridge + smooth) ------------------
def build_adaptive_nnls_exog_smooth(methods_val, methods_test, df_true, start_ds, end_ds, win=6, alpha=0.0, beta=0.0):
    future_idx = pd.date_range(start_ds, end_ds, freq="MS")
    names = list(methods_test.keys())
    out = pd.DataFrame({"ds": future_idx})
    preds = {m: pd.concat([methods_val[m][["ds","orders","stock"]],
                           methods_test[m][["ds","orders","stock"]]]).reset_index(drop=True) for m in names}
    combined_orders, combined_stock = [], []
    w_prev_o, w_prev_s = None, None
    for ds in future_idx:
        end_w = ds - pd.DateOffset(months=1)
        hist = df_true[(df_true["ds"]>=VAL_START)&(df_true["ds"]<=end_w)][["ds","orders","stock"]]
        hist = hist.tail(win)
        if len(hist)==0:
            w_o = np.ones(len(names))/len(names)
            w_s = np.ones(len(names))/len(names)
        else:
            A_o = np.vstack([preds[m].merge(hist[["ds"]], on="ds", how="inner")["orders"].to_numpy() for m in names]).T
            A_s = np.vstack([preds[m].merge(hist[["ds"]], on="ds", how="inner")["stock"].to_numpy()  for m in names]).T
            y_o = hist["orders"].to_numpy(); y_s = hist["stock"].to_numpy()
            w_o = nnls_adaptive_update(A_o, y_o, w_prev_o, alpha=alpha, beta=beta, iters=400)
            w_s = nnls_adaptive_update(A_s, y_s, w_prev_s, alpha=alpha, beta=beta, iters=400)
        cur = {m: methods_test[m][methods_test[m]["ds"]==ds] for m in names}
        ord_val = sum(float(w_o[i]) * float(cur[names[i]]["orders"].values[0]) for i in range(len(names)))
        stc_val = sum(float(w_s[i]) * float(cur[names[i]]["stock"].values[0])  for i in range(len(names)))
        combined_orders.append(ord_val); combined_stock.append(stc_val)
        w_prev_o, w_prev_s = w_o, w_s
    out["orders"] = combined_orders; out["stock"] = combined_stock
    return _postprocess_exog(out)

for w in ADAPT_WINS:
    tag = f"Adaptive5-NNLS-w{w}"
    adaptive_full  = build_adaptive_nnls_exog_smooth({m:exog_val[m] for m in ALL5},
                                                     {m:exog_test_full[m] for m in ALL5},
                                                     df_raw[["ds","orders","stock"]],
                                                     TEST_START, TEST_END, win=w,
                                                     alpha=RIDGE_ALPHA, beta=SMOOTH_BETA)
    adaptive_short = adaptive_full[adaptive_full["ds"]<=TEST_END_SHORT].copy()
    exog_test_full[tag]  = adaptive_full
    exog_test_short[tag] = adaptive_short

# VAL benzeri rezidü haritaları (PI için)
def get_val_residuals(exog_tbl):
    rep,_ = eval_exog_on_val(exog_tbl)
    return rep

val_resid_map = {}
for tag, ex in {**exog_val, **tag_ex_val, **tag_nnls}.items():
    val_resid_map[tag] = eval_exog_on_val(ex)[0]
for w in ADAPT_WINS:
    tag = f"Adaptive5-NNLS-w{w}"
    val_resid_map[tag] = val_resid_map.get("ML-Exog XGB", list(val_resid_map.values())[0])

# ------------------ Predict & evaluate on TEST (ALL exogs × Y variants) ------------------
"""
def add_bootstrap_intervals(pred_df, residuals, B=B_BOOT):
    res = np.array(residuals, dtype=float)
    if res.size == 0 or not np.any(np.isfinite(res)):
        scale = max(1e-6, np.median(np.abs(pred_df["yhat"].values)) * 0.1)
        sims = pred_df["yhat"].to_numpy().reshape(-1,1) + rng_boot.laplace(0.0, scale, size=(len(pred_df), B))
    else:
        res = res[np.isfinite(res)]
        idx = rng_boot.integers(0, res.size, size=(len(pred_df), B))
        noise = res[idx]; sims = pred_df["yhat"].to_numpy().reshape(-1,1) + noise
    lo80 = np.nanquantile(sims, 0.10, axis=1)
    hi80 = np.nanquantile(sims, 0.90, axis=1)
    lo95 = np.nanquantile(sims, 0.025, axis=1)
    hi95 = np.nanquantile(sims, 0.975, axis=1)
    out = pred_df.copy()
    out["pi80_lo"] = lo80; out["pi80_hi"] = hi80
    out["pi95_lo"] = lo95; out["pi95_hi"] = hi95
    return out, sims
"""
def add_bootstrap_intervals(pred_df, residuals, B=B_BOOT, mode="auto"):
    """
    mode: "auto" | "smooth" | "parametric"
    - auto: residual havuzu küçükse smooth'a geçer
    - smooth: smoothed bootstrap (res + jitter)
    - parametric: Laplace(0, b) ile param. bootstrap
    """
    yhat = pred_df["yhat"].to_numpy().reshape(-1,1)

    res = np.asarray(residuals, float)
    res = res[np.isfinite(res)]
    n = res.size
    # robust ölçek (MAD)
    if n > 0:
        mad = np.median(np.abs(res - np.median(res)))
    else:
        mad = np.nan
    b_lap = (mad/np.sqrt(2)) if (mad > 0) else (np.std(res) if n>1 else 1.0)

    if mode == "auto":
        # havuz çok küçükse/sınırlıysa smooth'a kay
        use_smooth = (n < 24) or (len(np.unique(np.round(res, 6))) <= 8)
        mode = "smooth" if use_smooth else "resample"

    if mode == "parametric":
        # saf param. Laplace noise
        noise = rng_boot.laplace(0.0, max(b_lap, 1e-6), size=(len(pred_df), B))

    elif mode == "smooth":
        # smoothed bootstrap: resample + küçük jitter
        if n == 0:
            noise = rng_boot.laplace(0.0, 0.1*np.nan_to_num(b_lap, nan=1.0), size=(len(pred_df), B))
        else:
            idx = rng_boot.integers(0, n, size=(len(pred_df), B))
            base = res[idx]
            jitter = rng_boot.laplace(0.0, max(0.25*max(b_lap,1e-6), 1e-6), size=(len(pred_df), B))
            noise = base + jitter

    else:
        # klasik saf resample (büyük havuzlarda yeterli)
        if n == 0:
            noise = rng_boot.laplace(0.0, 1.0, size=(len(pred_df), B))
        else:
            idx = rng_boot.integers(0, n, size=(len(pred_df), B))
            noise = res[idx]

    sims = yhat + noise
    lo80 = np.nanquantile(sims, 0.10, axis=1)
    hi80 = np.nanquantile(sims, 0.90, axis=1)
    lo95 = np.nanquantile(sims, 0.025, axis=1)
    hi95 = np.nanquantile(sims, 0.975, axis=1)

    out = pred_df.copy()
    out["pi80_lo"] = lo80; out["pi80_hi"] = hi80
    out["pi95_lo"] = lo95; out["pi95_hi"] = hi95
    return out, sims


def infer_starting_stock(df_raw, test_start, override=None):
    if override is not None: return float(override)
    prev = df_raw[df_raw["ds"] < test_start].tail(1)
    if "stock" in prev.columns and len(prev):
        return float(max(0.0, prev["stock"].iloc[0]))
    return 0.0

def stockout_probability(starting_stock, sims_matrix):
    sims = np.maximum(0.0, sims_matrix)
    cum = np.cumsum(sims, axis=0)
    tts = np.full(sims.shape[1], np.nan)
    for b in range(sims.shape[1]):
        idx = np.where(cum[:,b] >= starting_stock)[0]
        if idx.size>0: tts[b] = idx[0] + 1
    p3 = np.mean(np.nan_to_num(tts, nan=np.inf) <= 3)
    p6 = np.mean(np.nan_to_num(tts, nan=np.inf) <= 6)
    exp_t = np.nanmean(tts) if np.any(~np.isnan(tts)) else np.nan
    return float(p3), float(p6), (None if np.isnan(exp_t) else float(exp_t))

def cum_demand_quantile(sims_matrix, months, q=0.5):
    months = int(months)
    if months <= 0: return 0.0
    sims = np.maximum(0.0, sims_matrix)
    sums = np.sum(sims[:months, :], axis=0)
    return float(np.nanquantile(sums, q))

def round_moq_lot(qty, moq=MOQ, lot=LOT_SIZE):
    q = max(0.0, qty)
    if q <= 0: return 0.0
    q = max(q, moq)
    if lot > 1e-9:
        q = np.ceil(q / lot) * lot
    return float(q)

hist_min = trainval_df[["ds","y","orders","stock","month","year"]].copy()

def predict_with_variant(exog_table, variant, val_weights):
    if variant == "RF":
        p,_ = recursive_forward_predict_y(rf_model,  FEATURES_Y, hist_min.copy(), exog_table, TEST_START, exog_table["ds"].max())
    elif variant == "XGB":
        p,_ = recursive_forward_predict_y(xgb_model, FEATURES_Y, hist_min.copy(), exog_table, TEST_START, exog_table["ds"].max())
    else:
        prf,_  = recursive_forward_predict_y(rf_model,  FEATURES_Y, hist_min.copy(), exog_table, TEST_START, exog_table["ds"].max())
        pxgb,_ = recursive_forward_predict_y(xgb_model, FEATURES_Y, hist_min.copy(), exog_table, TEST_START, exog_table["ds"].max())
        p = prf.merge(pxgb, on="ds", suffixes=("_rf","_xgb"))
        w_rf, w_xgb = val_weights
        p["yhat"] = w_rf*p["yhat_rf"] + w_xgb*p["yhat_xgb"]
        p = p[["ds","yhat"]]
    return p

VARIANTS = ["RF","XGB","Y-ENS"]

ALL_EXOG_METHODS = BASIC + [
    "All-5-INV","Top2-INV","Top3-INV",
    "All-5-NNLS","Top2-NNLS","Top3-NNLS",
    "All-5-NNLS-Ridge","Top2-NNLS-Ridge","Top3-NNLS-Ridge",
] + [f"Adaptive5-NNLS-w{w}" for w in ADAPT_WINS]

results_rows = []

for horizon, ds_end, pool in [("Full", TEST_END, exog_test_full),
                              ("Short3", TEST_END_SHORT, exog_test_short)]:
    for m in ALL_EXOG_METHODS:
        ex_tbl = pool[m]
        val_rep_m = val_resid_map.get(m, val_rep["ML-Exog XGB"])
        for var in VARIANTS:
            preds = predict_with_variant(ex_tbl, var, val_rep_m["weights"])
            resids = np.array(val_rep_m["residuals"]["ENS" if var=="Y-ENS" else var], dtype=float)
            preds_pi, sims = add_bootstrap_intervals(preds, resids, B=B_BOOT)
            truth = df_raw[(df_raw["ds"]>=TEST_START)&(df_raw["ds"]<=ds_end)][["ds","y"]]
            eval_df = truth.merge(preds_pi, on="ds", how="left")
            mae, rmse, mape = mae_rmse_mape(eval_df["y"], eval_df["yhat"])
            starting_stock = infer_starting_stock(df_raw, TEST_START, STARTING_STOCK_OVERRIDE)
            p3, p6, exp_t = stockout_probability(starting_stock, sims)
            out_path = f"outputs/preds_{horizon}_{m}_{var}.csv".replace(' ','_')
            eval_df.to_csv(out_path, index=False)
            wrf, wxgb = val_rep_m["weights"]
            results_rows.append([horizon, m, var, mae, rmse, mape, wrf if var=="Y-ENS" else np.nan,
                                 wxgb if var=="Y-ENS" else np.nan, p3, p6, exp_t])

summary_cols = ["Horizon","Exog","Y-Variant","MAE","RMSE","MAPE","w_RF","w_XGB","P_stockout_3m","P_stockout_6m","E_T_stockout_mo"]
summary_all = pd.DataFrame(results_rows, columns=summary_cols).sort_values(["Horizon","Exog","Y-Variant"])
summary_all.to_csv("outputs/test_summary_ALL_with_NNLS_Ridge_adaptive.csv", index=False)

print("\n=== TEST Summary — ALL Exogs × Y Variants (Inverse vs NNLS vs Ridge vs Adaptive) ===")
print(summary_all.to_string(index=False))

# ------------------ Best combo & OMS scenarios ------------------
best_mask = (summary_all["Horizon"]=="Full")
best_row = summary_all.loc[best_mask].sort_values("MAE").iloc[0]
BEST_EXOG = best_row["Exog"]; BEST_YVAR = best_row["Y-Variant"]

sel_path = f"outputs/preds_Full_{BEST_EXOG}_{BEST_YVAR}.csv".replace(' ','_')
best_eval = pd.read_csv(sel_path, parse_dates=["ds"])
ex_tbl = exog_test_full[BEST_EXOG]
val_rep_m = val_resid_map.get(BEST_EXOG, val_rep["ML-Exog XGB"])
preds = predict_with_variant(ex_tbl, BEST_YVAR, val_rep_m["weights"])
resids = np.array(val_rep_m["residuals"]["ENS" if BEST_YVAR=="Y-ENS" else BEST_YVAR], dtype=float)
preds_pi, sims = add_bootstrap_intervals(preds, resids, B=B_BOOT)

starting_stock = infer_starting_stock(df_raw, TEST_START, STARTING_STOCK_OVERRIDE)

# Senaryo tablosu
sc_rows = []
for tchk in T_CHECK_LIST:
    for hcov in H_COVER_LIST:
        for q in COVER_Q_LIST:
            p3, p6, exp_t = stockout_probability(starting_stock, sims)
            need_reorder = (exp_t is not None) and (exp_t <= tchk)
            cum_need = cum_demand_quantile(sims, hcov, q=q)
            raw_qty = max(0.0, cum_need - starting_stock) if need_reorder else 0.0
            ord_qty = round_moq_lot(raw_qty, moq=MOQ, lot=LOT_SIZE)
            sc_rows.append([tchk, hcov, q, starting_stock, p3, p6, exp_t, cum_need, raw_qty, ord_qty, need_reorder])

sc_cols = ["T_CHECK","H_COVER","Q","Starting_Stock","P_stockout_3m","P_stockout_6m","E_T_stockout_mo",
           "CumDemand_Q","OrderQty_raw","OrderQty_rounded","NeedReorder"]
sc_df = pd.DataFrame(sc_rows, columns=sc_cols)
sc_df.to_csv("outputs/oms_scenarios.csv", index=False)

print("\n=== OMS Scenarios (parametric; MOQ/LOT uygulanmış) ===")
print(sc_df.to_string(index=False))

rec_row = sc_df.iloc[0]
rec = {
    "selected_combo": {"exog": BEST_EXOG, "y_variant": BEST_YVAR},
    "starting_stock": starting_stock,
    "scenario": {"T_CHECK": float(rec_row["T_CHECK"]), "H_COVER": float(rec_row["H_COVER"]), "Q": float(rec_row["Q"]),
                 "MOQ": MOQ, "LOT_SIZE": LOT_SIZE},
    "stockout_probs": {"p_3m": float(rec_row["P_stockout_3m"]), "p_6m": float(rec_row["P_stockout_6m"]),
                       "E_T_mo": (None if pd.isna(rec_row["E_T_stockout_mo"]) else float(rec_row["E_T_stockout_mo"]))},
    "coverage_need": {"cum_demand_quantile": float(rec_row["CumDemand_Q"])},
    "recommendation": {"need_reorder": bool(rec_row["NeedReorder"]), "order_qty_raw": float(rec_row["OrderQty_raw"]),
                       "order_qty_rounded": float(rec_row["OrderQty_rounded"])}
}
with open("outputs/reorder_recommendation.json","w",encoding="utf-8") as f:
    json.dump(rec, f, ensure_ascii=False, indent=2)

print("\n=== OMS Recommendation (selected best combo & first scenario) ===")
print(f"Selected combo: Exog={BEST_EXOG} | Y={BEST_YVAR}")
print(f"Starting stock: {starting_stock:.2f}")
print(f"Stockout: P<=3m={rec_row['P_stockout_3m']:.3f}, P<=6m={rec_row['P_stockout_6m']:.3f}, "
      f"E[T_stockout]={rec_row['E_T_stockout_mo'] if not pd.isna(rec_row['E_T_stockout_mo']) else 'NA'} months")
print(f"Policy: if E[T] <= {rec_row['T_CHECK']} months -> cover {rec_row['H_COVER']} months at q={rec_row['Q']}")
print(f"Cumulative demand needed: {rec_row['CumDemand_Q']:.2f}")
if rec_row["NeedReorder"]:
    print(f"REORDER NEEDED → Order Qty (raw) ≈ {rec_row['OrderQty_raw']:.2f} → Rounded (MOQ={MOQ}, LOT={LOT_SIZE}) ⇒ {rec_row['OrderQty_rounded']:.2f}")
else:
    print("No reorder needed under current scenario.")

# ------------------ SHAP / Feature Importances ------------------
def dump_feature_importances(model, model_name, X_df):
    out = pd.DataFrame({"feature": X_df.columns, "importance": getattr(model, "feature_importances_", np.zeros(X_df.shape[1]))})
    out = out.sort_values("importance", ascending=False)
    out.to_csv(f"outputs/feat_importance_{model_name}.csv", index=False)
    return out

def dump_shap_values(model, model_name, X_df):
    if not HAVE_SHAP:
        return None
    try:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_df)
        sv_arr = np.array(sv)
        if sv_arr.ndim == 3:
            sv_arr = np.mean(np.abs(sv_arr), axis=0)
        mean_abs = np.mean(np.abs(sv_arr), axis=0)
        out = pd.DataFrame({"feature": X_df.columns, "mean_abs_shap": mean_abs}).sort_values("mean_abs_shap", ascending=False)
        out.to_csv(f"outputs/shap_importance_{model_name}.csv", index=False)
        return out
    except Exception:
        return None

X_trainval = trainval_df[FEATURES_Y].copy()
dump_feature_importances(rf_model,  "rf",  X_trainval)
dump_feature_importances(xgb_model, "xgb", X_trainval)
dump_shap_values(rf_model,  "rf",  X_trainval)
dump_shap_values(xgb_model, "xgb", X_trainval)

# ------------------ Quick plot (best combo) ------------------
def plot_with_pi(eval_df, title, savepath):
    plt.figure(figsize=(12,6))
    plt.plot(eval_df["ds"], eval_df["y"], "o-", label="Gerçek")
    plt.plot(eval_df["ds"], eval_df["yhat"], "--", label="Tahmin")
    if {"pi80_lo","pi80_hi"}.issubset(eval_df.columns):
        plt.fill_between(eval_df["ds"], eval_df["pi80_lo"], eval_df["pi80_hi"], alpha=0.25, label="PI 80%")
    if {"pi95_lo","pi95_hi"}.issubset(eval_df.columns):
        plt.fill_between(eval_df["ds"], eval_df["pi95_lo"], eval_df["pi95_hi"], alpha=0.15, label="PI 95%")
    plt.title(title); plt.xlabel("Tarih"); plt.ylabel("Satış"); plt.legend(); plt.tight_layout()
    plt.savefig(savepath, dpi=160); plt.close()

if os.path.exists(sel_path):
    dfp = pd.read_csv(sel_path, parse_dates=["ds"])
    plot_with_pi(dfp, f"{BEST_EXOG} • {BEST_YVAR} • Full", f"outputs/plot_full_{BEST_EXOG}_{BEST_YVAR}.png".replace(' ','_'))

print("\nArtifacts:")
print("- outputs/val_exog_selection_basic.csv")
print("- outputs/val_exog_selection_ALL_with_NNLS_Ridge.csv")
print("- outputs/test_summary_ALL_with_NNLS_Ridge_adaptive.csv")
print("- outputs/oms_scenarios.csv  (OMS senaryo tablosu)")
print("- outputs/preds_<Horizon>_<Exog>_<Variant>.csv")
print("- outputs/reorder_recommendation.json  (OMS sipariş önerisi)")
print("- outputs/plot_full_<SelectedExog>_<Y>.png")
print("- outputs/feat_importance_*.csv / shap_importance_*.csv")


23:00:31 - cmdstanpy - INFO - Chain [1] start processing
23:00:31 - cmdstanpy - INFO - Chain [1] done processing
23:00:31 - cmdstanpy - INFO - Chain [1] start processing



=== ROCV Best Params ===
RF : {'n_estimators': 400, 'max_depth': 12, 'min_samples_split': 2, 'min_samples_leaf': 1} | ROCV_MAE=71.55
XGB: {'n_estimators': 800, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0} | ROCV_MAE=59.43


23:00:32 - cmdstanpy - INFO - Chain [1] done processing
23:00:47 - cmdstanpy - INFO - Chain [1] start processing
23:00:47 - cmdstanpy - INFO - Chain [1] done processing
23:00:57 - cmdstanpy - INFO - Chain [1] start processing
23:00:58 - cmdstanpy - INFO - Chain [1] done processing
23:01:06 - cmdstanpy - INFO - Chain [1] start processing
23:01:06 - cmdstanpy - INFO - Chain [1] done processing
23:01:16 - cmdstanpy - INFO - Chain [1] start processing
23:01:16 - cmdstanpy - INFO - Chain [1] done processing



=== VAL Exog Selection (BASIC methods) ===
       Exog  VAL_MAE_RF  VAL_MAE_XGB  VAL_MAE_YENS  VAL_RMSE_YENS  VAL_MAPE_YENS     w_RF    w_XGB
ML-Exog XGB  113.313021   108.419879    110.812459     123.732288      83.635409 0.488966 0.511034
 ML-Exog RF  112.138021   110.933634    111.532576     124.347148      84.627940 0.497300 0.502700
     SARIMA  112.883125   117.380941    115.088104     128.484952      83.527956 0.509767 0.490233
    Prophet  117.298542   125.187277    121.114588     139.463520      86.428829 0.516266 0.483734
   Ensemble  124.168646   126.224525    125.188146     139.117135     105.982818 0.504105 0.495895
        ETS  121.849062   163.542419    135.560694     154.890977     128.393360 0.573046 0.426954


23:01:26 - cmdstanpy - INFO - Chain [1] start processing
23:01:26 - cmdstanpy - INFO - Chain [1] done processing
23:01:26 - cmdstanpy - INFO - Chain [1] start processing



=== VAL Exog Selection — Inverse + NNLS (Ridge dahil) ===
            Exog  VAL_MAE_RF  VAL_MAE_XGB  VAL_MAE_YENS  VAL_RMSE_YENS  VAL_MAPE_YENS     w_RF    w_XGB
     ML-Exog XGB  113.313021   108.419879    110.812459     123.732288      83.635409 0.488966 0.511034
       All-5-INV  112.192708   109.722127    110.943665     123.161337      82.489558 0.494433 0.505567
        Top2-INV  113.516771   108.950562    111.186805     123.192233      83.603291 0.489737 0.510263
        Top3-INV  112.997500   109.680723    111.314410     123.926259      84.326946 0.492553 0.507447
       Top2-NNLS  113.361354   109.757367    111.530254     123.332109      84.028563 0.491924 0.508076
       Top3-NNLS  113.361354   109.757367    111.530254     123.332109      84.028563 0.491924 0.508076
 Top2-NNLS-Ridge  113.361354   109.757367    111.530254     123.332109      84.028563 0.491924 0.508076
 Top3-NNLS-Ridge  113.361354   109.757367    111.530254     123.332109      84.028563 0.491924 0.508076
     

23:01:26 - cmdstanpy - INFO - Chain [1] done processing
23:01:26 - cmdstanpy - INFO - Chain [1] start processing
23:01:26 - cmdstanpy - INFO - Chain [1] done processing
23:01:26 - cmdstanpy - INFO - Chain [1] start processing
23:01:27 - cmdstanpy - INFO - Chain [1] done processing
23:01:58 - cmdstanpy - INFO - Chain [1] start processing
23:01:59 - cmdstanpy - INFO - Chain [1] done processing
23:02:07 - cmdstanpy - INFO - Chain [1] start processing
23:02:07 - cmdstanpy - INFO - Chain [1] done processing
23:02:15 - cmdstanpy - INFO - Chain [1] start processing
23:02:15 - cmdstanpy - INFO - Chain [1] done processing
23:02:23 - cmdstanpy - INFO - Chain [1] start processing
23:02:23 - cmdstanpy - INFO - Chain [1] done processing
23:02:30 - cmdstanpy - INFO - Chain [1] start processing
23:02:30 - cmdstanpy - INFO - Chain [1] done processing
23:02:38 - cmdstanpy - INFO - Chain [1] start processing
23:02:38 - cmdstanpy - INFO - Chain [1] done processing
23:02:47 - cmdstanpy - INFO - Chain [1] 


=== TEST Summary — ALL Exogs × Y Variants (Inverse vs NNLS vs Ridge vs Adaptive) ===
Horizon              Exog Y-Variant        MAE       RMSE      MAPE     w_RF    w_XGB  P_stockout_3m  P_stockout_6m  E_T_stockout_mo
   Full Adaptive5-NNLS-w3        RF  62.364375  72.363908 54.423296      NaN      NaN        0.99125        1.00000         1.611250
   Full Adaptive5-NNLS-w3       XGB  73.188832  91.839064 67.558207      NaN      NaN        1.00000        1.00000         1.533750
   Full Adaptive5-NNLS-w3     Y-ENS  67.896039  81.838126 61.135680 0.488966 0.511034        0.99875        1.00000         1.572500
   Full Adaptive5-NNLS-w4        RF  62.803750  73.107397 55.005948      NaN      NaN        0.99000        1.00000         1.611250
   Full Adaptive5-NNLS-w4       XGB  72.407351  90.920258 66.736935      NaN      NaN        1.00000        1.00000         1.553750
   Full Adaptive5-NNLS-w4     Y-ENS  67.711515  81.771053 61.000880 0.488966 0.511034        0.99500        1.00000 